In [2]:
import os
import glob
import pandas as pd
from tqdm import tqdm

def build_deseq2_matrix(data_dir):
    print("=== Starting Unified Matrix Generation for DESeq2 ===")
    
    # 1. Automatically locate the sample sheet
    sheet_patterns = [
        os.path.join(data_dir, "**", "*sheet*.tsv"),
        os.path.join(data_dir, "*.tsv")
    ]
    
    found_sheets = []
    for pattern in sheet_patterns:
        found_sheets.extend(glob.glob(pattern, recursive=True))
        
    # Remove duplicates and ensure we don't accidentally grab a gene count file as a sheet
    found_sheets = list(set([s for s in found_sheets if "star_gene_counts" not in s.lower()]))
    
    if not found_sheets:
        print("❌ Error: No .tsv sample sheets found in the data directory.")
        return
        
    print(f"📋 Found {len(found_sheets)} potential metadata sheets:")
    for s in found_sheets:
        print(f"   -> {os.path.basename(s)}")

    # 2. Merge all sample sheet metadata maps together
    file_metadata_map = {}
    for sheet_path in found_sheets:
        try:
            df_sheet = pd.read_csv(sheet_path, sep='\t')
            df_sheet.columns = [col.strip() for col in df_sheet.columns]
            
            # Ensure required columns exist
            if 'File Name' in df_sheet.columns and 'Sample ID' in df_sheet.columns:
                for _, row in df_sheet.iterrows():
                    filename = str(row['File Name']).strip()
                    file_metadata_map[filename] = str(row['Sample ID']).strip()
        except Exception as e:
            print(f"⚠️ Skipping sheet {os.path.basename(sheet_path)}: {e}")

    print(f"✅ Loaded {len(file_metadata_map)} patient-file mappings from sheets.")

    # 3. Find every STAR count file physically present in the folders recursively
    file_pattern = os.path.join(data_dir, "**", "*.rna_seq.augmented_star_gene_counts.tsv")
    all_physical_files = glob.glob(file_pattern, recursive=True)

    if not all_physical_files:
        print("❌ Error: No RNA-Seq count files found in the directory.")
        return

    print(f"📂 Detected {len(all_physical_files)} total physical STAR count files.")

    # 4. Extract RAW counts for DESeq2
    data_frames = []
    gene_info = None
    skipped_count = 0
    
    for file_path in tqdm(all_physical_files, desc="Parsing Files"):
        filename = os.path.basename(file_path)
        
        # Check against metadata sheets map with fuzzy fallback
        if filename in file_metadata_map:
            sample_id = file_metadata_map[filename]
        else:
            # Fuzzy match fallback
            fuzzy_match = [k for k in file_metadata_map.keys() if k in filename or filename in k]
            if fuzzy_match:
                sample_id = file_metadata_map[fuzzy_match[0]]
            else:
                # If completely unmatched, use the filename as the column header
                sample_id = filename.split('.')[0]
                skipped_count += 1
                
        try:
            # Read file, ignoring the GENCODE version comment at the top
            df = pd.read_csv(file_path, sep='\t', comment='#')
            
            # Remove tracking/statistical rows (N_unmapped, etc.)
            df = df[~df['gene_id'].astype(str).str.startswith('N_')]
            
            # Save the gene names mapping from the very first file
            if gene_info is None:
                gene_info = df[['gene_id', 'gene_name']].copy()
                gene_info.set_index('gene_id', inplace=True)
            
            # Extract ONLY the raw, unnormalized integer counts for DESeq2
            counts_df = df[['gene_id', 'unstranded']].copy()
            counts_df.rename(columns={'unstranded': sample_id}, inplace=True)
            counts_df.set_index('gene_id', inplace=True)
            
            data_frames.append(counts_df)
            
        except Exception as e:
            print(f"⚠️ Error parsing file {filename}: {e}")

    if skipped_count > 0:
        print(f"⏭️ {skipped_count} files didn't perfectly match the sample sheet (used filenames instead).")

    # 5. Merge all files into the final DESeq2 Matrix
    print("Merging all samples into one master DESeq2 matrix...")
    master_counts = pd.concat(data_frames, axis=1)

    # Attach the Gene Names back to the matrix
    final_matrix = gene_info.join(master_counts)
    final_matrix.reset_index(inplace=True)
    
    # Rename columns to perfectly match the E-MTAB-5244 structure
    final_matrix.rename(columns={'gene_id': 'Gene ID', 'gene_name': 'Gene Name'}, inplace=True)

    # Save to directory
    out_path = os.path.join(data_dir, 'TCGA_Raw_Counts_for_DESeq2.tsv')
    final_matrix.to_csv(out_path, sep='\t', index=False)

    print(f"🎉 Run complete! Matrix created: {out_path}")
    print(f"   -> Contains {final_matrix.shape[0]} genes across {len(data_frames)} samples.")


# =====================================================================
# EXECUTION
# =====================================================================
GDC_DATA_DIRECTORY = r"C:\Users\samal\Downloads\Abhinaya"
build_deseq2_matrix(GDC_DATA_DIRECTORY)

=== Starting Unified Matrix Generation for DESeq2 ===
📋 Found 9 potential metadata sheets:
   -> gdc_sample_sheet.2026-06-24 (4).tsv
   -> gdc_sample_sheet.2026-06-24 (8).tsv
   -> gdc_sample_sheet.2026-06-24 (2).tsv
   -> tumour_DESeq2.tsv
   -> gdc_sample_sheet.2026-06-24 (7).tsv
   -> gdc_sample_sheet.2026-06-24 (1).tsv
   -> gdc_sample_sheet.2026-06-24 (3).tsv
   -> gdc_sample_sheet.2026-06-24 (5).tsv
   -> gdc_sample_sheet.2026-06-24 (6).tsv
✅ Loaded 1331 patient-file mappings from sheets.
📂 Detected 41 total physical STAR count files.


Parsing Files: 100%|███████████████████████████████████████████████████████████████████| 41/41 [00:02<00:00, 16.49it/s]


Merging all samples into one master DESeq2 matrix...
🎉 Run complete! Matrix created: C:\Users\samal\Downloads\Abhinaya\TCGA_Raw_Counts_for_DESeq2.tsv
   -> Contains 60660 genes across 41 samples.
